# Tarang ECG Classifier v4
**Changes from v3_final → v4:**
- **RR normalization fix**: RR_MEAN/RR_STD computed right after train split; all train/val/test/quantization use consistently normalized RR features — eliminates train/inference mismatch
- **Training stability**: Gradient clipping (`clipnorm=1.0`), 2-epoch LR warmup, cosine decay schedule — eliminates periodic collapse to degenerate "always predict S" state
- **Focal loss rebalanced**: alpha `[0.15, 0.45, 0.40]` (was `[0.10, 0.60, 0.30]`), S augmentation 10× (was 15×) — v3's double-aggressive settings caused S F1 *regression* from 0.16→0.10
- **Checkpoint on macro F1**: Custom `MacroF1Callback` replaces val_loss monitoring — ensures best checkpoint reflects S/V quality, not N-dominated loss
- **SVDB channel audit**: Diagnostic cell logs which SVDB records use single-lead duplicate-channel fallback
- **RR branch dropout**: `Dropout(0.2)` added between Dense(16) and Dense(8) for regularization
- All prior v3 features retained: 187-sample window, dual-input ECG+RR, Block 4 at 64ch, SVDB integration

**SVDB download (run once before Cell 1):**
```bash
import wfdb
wfdb.dl_database('svdb', dl_dir='C:/MMD Public/Hackathons/Team Ocelleon/dataset/mit-bih-svdb-1.0.0')
```

In [ ]:
import os
import json
import subprocess
import warnings
from collections import Counter

import wfdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import regularizers

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score

warnings.filterwarnings('ignore')

# ── PATHS — edit these ────────────────────────────────────────────────────────
MITBIH_PATH = 'C:/MMD Public/Hackathons/Team Ocelleon/dataset/mit-bih-arrhythmia-database-1.0.0'
SVDB_PATH   = 'C:/MMD Public/Hackathons/Team Ocelleon/dataset/mit-bih-svdb-1.0.0'

OUTPUTS_DIR      = 'outputs'
MODEL_CHECKPOINT = f'{OUTPUTS_DIR}/tarang_best.keras'
TFLITE_PATH      = f'{OUTPUTS_DIR}/tarang_int8.tflite'
HEADER_PATH      = f'{OUTPUTS_DIR}/tarang_model.h'
os.makedirs(OUTPUTS_DIR, exist_ok=True)

# ── WINDOW — DO NOT CHANGE without updating firmware ─────────────────────────
WINDOW_PRE  = 93
WINDOW_POST = 94
WINDOW_LEN  = WINDOW_PRE + WINDOW_POST   # 187 samples = ~519ms at 360Hz

FS = 360   # MIT-BIH and SVDB are both 360Hz

CONFIDENCE_THRESHOLD = 0.70

print("TensorFlow:", tf.__version__)
print("WFDB      :", wfdb.__version__)
print(f"Window    : {WINDOW_LEN} samples ({WINDOW_LEN/FS*1000:.1f}ms)")
print(f"MIT-BIH   : {MITBIH_PATH}")
print(f"SVDB      : {SVDB_PATH}  (exists={os.path.isdir(SVDB_PATH)})")
print(f"Outputs   : {OUTPUTS_DIR}/")

In [ ]:
# AAMI/ANSI EC57 — clinical standard, do not modify
BEAT_MAP = {
    'N': 'N', 'L': 'N', 'R': 'N', 'e': 'N', 'j': 'N',
    'A': 'S', 'a': 'S', 'J': 'S', 'S': 'S',
    'V': 'V', 'E': 'V', 'F': 'V',   # Fusion → V (conservative/pacemaker safety)
    '/': 'Q', 'f': 'Q', 'Q': 'Q',
}
CLASSES_TO_USE = ['N', 'V', 'S']

MITBIH_ALL_RECORDS = [
    100,101,102,103,104,105,106,107,108,109,
    111,112,113,114,115,116,117,118,119,121,
    122,123,124,200,201,202,203,205,207,208,
    209,210,212,213,214,215,217,219,220,221,
    222,223,228,230,231,232,233,234
]

# Patient-wise test/val split — NEVER seen during training
MITBIH_TEST_RECORDS = [
    101,106,108,109,112,114,115,116,118,119,
    201,202,203,205,207,208,209,210,217,219,
    221,223,228,231,233,234
]
MITBIH_VAL_RECORDS   = [105, 124, 214, 220]
MITBIH_TRAIN_RECORDS = [r for r in MITBIH_ALL_RECORDS
                        if r not in MITBIH_TEST_RECORDS
                        and r not in MITBIH_VAL_RECORDS]

# SVDB: 78 records (e800–e877) — train only, no test leak
SVDB_RECORDS = [f'e{i}' for i in range(800, 878)]

print(f"MIT-BIH train : {len(MITBIH_TRAIN_RECORDS)} records")
print(f"MIT-BIH val   : {len(MITBIH_VAL_RECORDS)} records")
print(f"MIT-BIH test  : {len(MITBIH_TEST_RECORDS)} records (held out)")
print(f"SVDB train    : {len(SVDB_RECORDS)} records (S-class boost)")

In [ ]:
record     = wfdb.rdrecord(f'{MITBIH_PATH}/100')
annotation = wfdb.rdann(f'{MITBIH_PATH}/100', 'atr')

print("=== RECORD 100 ===")
print(f"Signals : {record.sig_name}")
print(f"Fs      : {record.fs} Hz")
print(f"Length  : {record.sig_len} samples ({record.sig_len/record.fs/60:.1f} min)")
print(f"Beats   : {len(annotation.sample)}")

signal = record.p_signal[:, 0]
fig, axes = plt.subplots(2, 1, figsize=(15, 7))

t = np.arange(0, 10*FS) / FS
axes[0].plot(t, signal[:10*FS], 'b-', linewidth=0.8)
r_in_win = [p for p in annotation.sample if p < 10*FS]
axes[0].scatter([p/FS for p in r_in_win], [signal[p] for p in r_in_win],
                color='red', s=40, zorder=5, label='R peaks')
axes[0].set_title('Raw ECG - Record 100 (10 seconds)')
axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('Amplitude (mV)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

center = r_in_win[3]
t2 = np.arange(-WINDOW_PRE, WINDOW_POST) / FS * 1000
axes[1].plot(t2, signal[center-WINDOW_PRE:center+WINDOW_POST], 'g-', linewidth=1.5)
axes[1].axvline(0, color='red', alpha=0.7, linestyle='--', label='R peak')
axes[1].set_title(f'Single beat window ({WINDOW_LEN} samples = {WINDOW_LEN/FS*1000:.0f}ms) — firmware match')
axes[1].set_xlabel('ms from R peak'); axes[1].set_ylabel('Amplitude (mV)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUTS_DIR}/ecg_exploration.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def rolling_window_normalize(signal, window_samples=10800):
    """
    Simulates firmware 30s Welford rolling normalization.
    Fast pandas version — milliseconds per record.
    window_samples = 10800 = 30s × 360Hz
    """
    s    = pd.Series(signal.astype(np.float64))
    roll = s.rolling(window=window_samples, min_periods=1)
    mean = roll.mean()
    std  = roll.std(ddof=0).fillna(0).clip(lower=1e-8)
    return ((s - mean) / std).values.astype(np.float32)


def compute_rr_features(peaks, beat_idx):
    """
    5 RR-interval features for one beat.
    peaks     : full sorted R-peak sample array for the record
    beat_idx  : index of current beat in peaks[]
    Returns float32 (5,): [rr_prev, rr_next, rr_ratio, rr_mean_5, rr_std_5] in seconds.
    Firmware replicates this from the Pan-Tompkins RR buffer.
    """
    n = len(peaks)
    i = beat_idx
    prev_idx = max(0, i-1)
    next_idx = min(n-1, i+1)
    rr_prev  = (peaks[i] - peaks[prev_idx]) / FS
    rr_next  = (peaks[next_idx] - peaks[i]) / FS
    rr_ratio = rr_prev / max(rr_next, 1e-4)

    lo = max(0, i-2); hi = min(n-1, i+2)
    local_rrs = np.diff(peaks[lo:hi+1]).astype(np.float32) / FS
    if len(local_rrs) == 0:
        rr_mean_5, rr_std_5 = rr_prev, 0.0
    else:
        rr_mean_5 = float(np.mean(local_rrs))
        rr_std_5  = float(np.std(local_rrs))

    return np.array([rr_prev, rr_next, rr_ratio, rr_mean_5, rr_std_5], dtype=np.float32)

print("rolling_window_normalize() and compute_rr_features() defined.")

In [ ]:
def load_records(db_path, record_list, source_tag, two_channel=True):
    """
    Load beats from a list of records.
    Returns: beats (N,187,2), rr (N,5), labels (N,), record_ids (N,)
    Single normalisation pass — no double norm, no uint16 round-trip.
    Also logs channel count per record for SVDB domain-gap diagnostics.
    """
    all_beats, all_rr, all_labels, all_recs = [], [], [], []
    channel_counts = {}  # record → n_channels (for SVDB audit)

    for rec_id in record_list:
        rec_str = str(rec_id)
        try:
            record     = wfdb.rdrecord(f'{db_path}/{rec_str}')
            annotation = wfdb.rdann(f'{db_path}/{rec_str}', 'atr')

            n_channels = record.p_signal.shape[1]
            channel_counts[rec_str] = n_channels

            # Single normalisation — windowed, no NLMS conversion for MIT-BIH/SVDB
            ecg_ch0 = rolling_window_normalize(record.p_signal[:, 0])
            if two_channel and n_channels > 1:
                ecg_ch1 = rolling_window_normalize(record.p_signal[:, 1])
            else:
                ecg_ch1 = ecg_ch0   # SVDB single-lead: duplicate channel

            all_peaks  = annotation.sample
            all_syms   = annotation.symbol
            valid_idxs = [i for i, s in enumerate(all_syms)
                          if BEAT_MAP.get(s, None) in CLASSES_TO_USE]

            beats_this = 0
            for idx in valid_idxs:
                peak   = all_peaks[idx]
                mapped = BEAT_MAP[all_syms[idx]]

                if peak - WINDOW_PRE < 0 or peak + WINDOW_POST >= len(ecg_ch0):
                    continue

                beat = np.stack([
                    ecg_ch0[peak - WINDOW_PRE : peak + WINDOW_POST],
                    ecg_ch1[peak - WINDOW_PRE : peak + WINDOW_POST],
                ], axis=-1).astype(np.float32)   # (187, 2)

                if np.any(np.isnan(beat)) or np.any(np.isinf(beat)):
                    continue

                rr = compute_rr_features(all_peaks, idx)

                all_beats.append(beat)
                all_rr.append(rr)
                all_labels.append(mapped)
                all_recs.append(f'{source_tag}_{rec_str}')
                beats_this += 1

            ch_tag = f'{n_channels}ch' + (' [DUP]' if n_channels == 1 else '')
            print(f"  {source_tag}/{rec_str}: {beats_this} beats  ({ch_tag})")

        except Exception as ex:
            print(f"  {source_tag}/{rec_str}: FAILED — {ex}")

    # Summary: channel distribution
    if channel_counts:
        n_single = sum(1 for v in channel_counts.values() if v == 1)
        n_multi  = sum(1 for v in channel_counts.values() if v > 1)
        print(f"\n  Channel summary: {n_multi} multi-lead, {n_single} single-lead (duplicate-channel fallback)")
        if n_single > 0:
            print(f"  WARNING: {n_single} records use duplicated ch0 — potential domain gap for minority classes")

    return (
        np.array(all_beats,  dtype=np.float32),
        np.array(all_rr,     dtype=np.float32),
        np.array(all_labels, dtype=object),
        np.array(all_recs,   dtype=object),
    )

print("load_records() defined.")

In [ ]:
print("Loading MIT-BIH Arrhythmia database...")
mb_beats, mb_rr, mb_labels, mb_recs = load_records(
    MITBIH_PATH, MITBIH_ALL_RECORDS, 'mitbih', two_channel=True
)
print(f"\nMIT-BIH: {mb_beats.shape}  classes={Counter(mb_labels)}")

In [ ]:
print("Loading SVDB (Supraventricular Arrhythmia Database)...")
svdb_ok = os.path.isdir(SVDB_PATH)

if svdb_ok:
    sv_beats, sv_rr, sv_labels, sv_recs = load_records(
        SVDB_PATH, SVDB_RECORDS, 'svdb', two_channel=False
    )
    print(f"\nSVDB: {sv_beats.shape}  classes={Counter(sv_labels)}")
else:
    print(f"SVDB not found at {SVDB_PATH}")
    print("Run: wfdb.dl_database('svdb', dl_dir=SVDB_PATH)  then re-run this cell.")
    print("Continuing with MIT-BIH only.")
    sv_beats  = np.empty((0, WINDOW_LEN, 2), dtype=np.float32)
    sv_rr     = np.empty((0, 5),             dtype=np.float32)
    sv_labels = np.array([], dtype=object)
    sv_recs   = np.array([], dtype=object)

In [ ]:
print("=" * 60)
print("SVDB CHANNEL AUDIT — Domain Gap Diagnostic")
print("=" * 60)
print()

if svdb_ok:
    single_lead_records = []
    multi_lead_records  = []
    for rec_id in SVDB_RECORDS:
        try:
            record = wfdb.rdrecord(f'{SVDB_PATH}/{rec_id}')
            n_ch = record.p_signal.shape[1]
            if n_ch == 1:
                single_lead_records.append(rec_id)
            else:
                multi_lead_records.append(rec_id)
        except Exception as ex:
            print(f"  {rec_id}: FAILED — {ex}")

    pct_single = len(single_lead_records) / max(len(SVDB_RECORDS), 1) * 100
    print(f"Multi-lead  : {len(multi_lead_records)} records")
    print(f"Single-lead : {len(single_lead_records)} records ({pct_single:.0f}%)")
    if single_lead_records:
        print(f"Single-lead IDs: {single_lead_records[:10]}{'...' if len(single_lead_records) > 10 else ''}")
        print()
        print("⚠  These records use the duplicate-channel fallback (ch1 = ch0).")
        print("   The model may learn S-class patterns from a different signal")
        print("   distribution than the MIT-BIH 2-lead test set.")
        if pct_single > 50:
            print("   CRITICAL: >50% of SVDB uses single-lead — consider")
            print("   dropping duplicated-channel records or training lead-invariant front end.")
    else:
        print("\n✓ All SVDB records are multi-lead — no domain gap from channel duplication.")
else:
    print("SVDB not loaded — skipping audit.")

In [ ]:
X_all      = np.concatenate([mb_beats, sv_beats], axis=0)
X_rr_all   = np.concatenate([mb_rr,    sv_rr],   axis=0)
y_raw_all  = np.concatenate([mb_labels, sv_labels])
recs_all   = np.concatenate([mb_recs,   sv_recs])

le    = LabelEncoder()
y_all = le.fit_transform(y_raw_all)
print("Label encoding:", {i: c for i, c in enumerate(le.classes_)})

test_mask  = np.array([
    r.startswith('mitbih_') and int(r.split('_')[1]) in MITBIH_TEST_RECORDS
    for r in recs_all])
val_mask   = np.array([
    r.startswith('mitbih_') and int(r.split('_')[1]) in MITBIH_VAL_RECORDS
    for r in recs_all])
train_mask = ~test_mask & ~val_mask   # MIT-BIH train + all SVDB

X_train,    X_val,    X_test    = X_all[train_mask],    X_all[val_mask],    X_all[test_mask]
X_rr_train, X_rr_val, X_rr_test = X_rr_all[train_mask], X_rr_all[val_mask], X_rr_all[test_mask]
y_train,    y_val,    y_test    = y_all[train_mask],    y_all[val_mask],    y_all[test_mask]

# ── RR NORMALIZATION — computed from clean train data BEFORE augmentation ────
# v3 bug: RR_MEAN/STD were computed in Cell 17 (after training), so training
# used raw RR (seconds) but INT8 quantization used normalized RR — mismatch.
# v4 fix: normalize everywhere consistently.
RR_MEAN = X_rr_train.mean(axis=0)
RR_STD  = X_rr_train.std(axis=0)
RR_STD[RR_STD < 1e-8] = 1e-8

feature_names = ['rr_prev','rr_next','rr_ratio','rr_mean_5','rr_std_5']
print("\nRR normalization stats (from train split):")
for i, name in enumerate(feature_names):
    print(f"  {name}: mean={RR_MEAN[i]:.5f}  std={RR_STD[i]:.5f}")

# Normalize train/val/test RR features
X_rr_train = ((X_rr_train - RR_MEAN) / RR_STD).astype(np.float32)
X_rr_val   = ((X_rr_val   - RR_MEAN) / RR_STD).astype(np.float32)
X_rr_test  = ((X_rr_test  - RR_MEAN) / RR_STD).astype(np.float32)

print(f"\nRR features normalized (zero-mean, unit-variance).")
print(f"  Train RR range: [{X_rr_train.min():.2f}, {X_rr_train.max():.2f}]")
print(f"  Val   RR range: [{X_rr_val.min():.2f}, {X_rr_val.max():.2f}]")
print(f"  Test  RR range: [{X_rr_test.min():.2f}, {X_rr_test.max():.2f}]")

print(f"\nX_all   : {X_all.shape}  (beats, {WINDOW_LEN}, 2ch)")
print(f"X_rr_all: {X_rr_all.shape}  (beats, 5 RR features)")
print(f"\nTrain: {len(X_train):,}  {Counter(le.inverse_transform(y_train))}")
print(f"Val  : {len(X_val):,}   {Counter(le.inverse_transform(y_val))}")
print(f"Test : {len(X_test):,}  {Counter(le.inverse_transform(y_test))}")

# Distribution plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2ecc71','#f39c12','#e74c3c']
cls_names = list(le.classes_)
for ax, ctr, title in zip(
        axes,
        [Counter(y_raw_all), Counter(le.inverse_transform(y_train))],
        ['Full dataset (MIT-BIH + SVDB)', 'Training split']):
    counts = [ctr.get(c, 0) for c in cls_names]
    ax.bar(cls_names, counts, color=colors)
    ax.set_title(title); ax.set_ylabel('Beats')
    for i, v in enumerate(counts):
        ax.text(i, v+100, f'{v:,}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUTPUTS_DIR}/class_distribution.png', dpi=100)
plt.show()

In [ ]:
def augment_batch(X_c, X_rr_c, class_name, n_copies):
    """
    Augment ECG (N,187,2) and RR features (N,5) consistently.
    RR features are already normalized (zero-mean, unit-variance) at this point.
    Returns aug_ecg (N*n_copies,187,2), aug_rr (N*n_copies,5).
    """
    n = len(X_c)
    ecg_list, rr_list = [], []
    t_arr = np.linspace(0, 2*np.pi, WINDOW_LEN)

    for _ in range(n_copies):
        a = X_c.copy()

        # Amplitude ±15%
        a *= np.random.uniform(0.85, 1.15, (n, 1, 1))

        # Baseline wander
        freqs  = np.random.uniform(0.3, 0.8, n)
        amps   = np.random.uniform(0.0, 0.12, n)
        wander = (amps[:, None] * np.sin(freqs[:, None] * t_arr[None, :]))[:, :, None]
        a += wander

        # Gaussian noise
        a += np.random.normal(0, np.random.uniform(0.01, 0.04), a.shape).astype(np.float32)

        # Time shift ±10 samples (tighter window = less shift vs v2's ±20)
        shifts = np.random.randint(-10, 11, n)
        for i in range(n):
            a[i] = np.roll(a[i], shifts[i], axis=0)

        # S: P-wave perturbation
        if class_name == 'S':
            a[:, :40, :] *= np.random.uniform(0.4, 1.6, (n, 1, 1))

        ecg_list.append(a.astype(np.float32))

        # RR feature augmentation — features are NORMALIZED (zero-mean, unit-var)
        # Perturbations are additive in z-score space
        rr = X_rr_c.copy()
        if class_name == 'S':
            # rr_prev shorter (premature) — shift down by 0.3–1.0 std
            rr[:, 0] -= np.random.uniform(0.3, 1.0, n)
            # rr_ratio: recompute as shifted prev relative to next
            rr[:, 2] += np.random.uniform(-0.5, 0.3, n)
            # rr_std_5 higher (more local variability) — shift up
            rr[:, 4] += np.random.uniform(0.2, 0.8, n)
        elif class_name == 'V':
            # compensatory pause — rr_next shifted up
            rr[:, 1] += np.random.uniform(0.1, 0.5, n)
            rr[:, 4] += np.random.uniform(0.0, 0.4, n)

        # Small global jitter in z-score space
        rr += np.random.normal(0, 0.1, rr.shape).astype(np.float32)
        rr  = np.clip(rr, -4.0, 4.0).astype(np.float32)
        rr_list.append(rr)

    return np.concatenate(ecg_list, axis=0), np.concatenate(rr_list, axis=0)


# v4: S=10 (was 15), V=6 — v3's S=15 + alpha=0.60 caused training collapse
# v2 had S=10 + alpha=0.50 and achieved S F1=0.16 (better than v3's 0.10)
AUG_COPIES = {'N': 0, 'V': 6, 'S': 10}

print("Augmenting minority classes...")
print(f"AUG_COPIES: {AUG_COPIES}")
ecg_aug_list = [X_train]
rr_aug_list  = [X_rr_train]
y_aug_list   = [y_train]

for ci, cname in enumerate(le.classes_):
    n_copies = AUG_COPIES.get(cname, 0)
    if n_copies == 0:
        continue
    mask = y_train == ci
    aug_ecg, aug_rr = augment_batch(X_train[mask], X_rr_train[mask], cname, n_copies)
    ecg_aug_list.append(aug_ecg)
    rr_aug_list.append(aug_rr)
    y_aug_list.append(np.tile(y_train[mask], n_copies))

X_train_aug    = np.concatenate(ecg_aug_list, axis=0)
X_rr_train_aug = np.concatenate(rr_aug_list,  axis=0)
y_train_aug    = np.concatenate(y_aug_list,   axis=0)

idx = np.random.permutation(len(X_train_aug))
X_train_aug    = X_train_aug[idx]
X_rr_train_aug = X_rr_train_aug[idx]
y_train_aug    = y_train_aug[idx]

print(f"\nBefore: {Counter(le.inverse_transform(y_train))}")
print(f"After : {Counter(le.inverse_transform(y_train_aug))}")
print(f"ECG   : {X_train_aug.shape}")
print(f"RR    : {X_rr_train_aug.shape}")

In [ ]:
def build_tarang_cnn_v4(ecg_shape=(WINDOW_LEN, 2), rr_shape=(5,), num_classes=3):
    """
    Dual-input ECG + RR model — v4.
    ECG branch: depthwise separable Conv1D × 4 blocks + GAP (~26KB params)
    RR branch:  Dense(16) → Dropout(0.2) → Dense(8)
    Merged:     Dense(32) → softmax
    Total ~30KB INT8 — well under EFR32 50KB limit.

    v4 change: Dropout(0.2) added in RR branch — prevents overfitting on
    now-normalized RR features which carry more discriminative signal.
    Block 4 (64ch, kernel 3) retained for S-class morphology discrimination.
    """
    ecg_in = tf.keras.Input(shape=ecg_shape, name='ecg_input')

    # Block 1 — local waveform / P-wave
    x = tf.keras.layers.SeparableConv1D(
        16, 7, padding='same', use_bias=False,
        depthwise_regularizer=regularizers.l2(1e-4))(ecg_in)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.MaxPooling1D(2)(x)
    x = tf.keras.layers.SpatialDropout1D(0.1)(x)

    # Block 2 — QRS morphology
    x = tf.keras.layers.SeparableConv1D(
        32, 5, padding='same', use_bias=False,
        depthwise_regularizer=regularizers.l2(1e-4))(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.MaxPooling1D(2)(x)
    x = tf.keras.layers.SpatialDropout1D(0.1)(x)

    # Block 3 — T wave / ST segment
    x = tf.keras.layers.SeparableConv1D(
        64, 5, padding='same', use_bias=False,
        depthwise_regularizer=regularizers.l2(1e-4))(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.SpatialDropout1D(0.15)(x)

    # Block 4 — S-class discrimination (restored — needed for subtle morphology)
    x = tf.keras.layers.SeparableConv1D(
        64, 3, padding='same', use_bias=False,
        depthwise_regularizer=regularizers.l2(1e-4))(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)

    x = tf.keras.layers.GlobalAveragePooling1D()(x)   # (64,)

    # RR branch — v4: Dropout(0.2) added for regularization
    rr_in = tf.keras.Input(shape=rr_shape, name='rr_input')
    r = tf.keras.layers.Dense(16, activation='relu',
                               kernel_regularizer=regularizers.l2(1e-4))(rr_in)
    r = tf.keras.layers.Dropout(0.2)(r)
    r = tf.keras.layers.Dense(8,  activation='relu',
                               kernel_regularizer=regularizers.l2(1e-4))(r)

    # Merge
    merged = tf.keras.layers.Concatenate()([x, r])   # (72,)
    merged = tf.keras.layers.Dense(32, use_bias=False)(merged)
    merged = tf.keras.layers.BatchNormalization()(merged)
    merged = tf.keras.layers.Activation('relu')(merged)
    merged = tf.keras.layers.Dropout(0.35)(merged)

    out = tf.keras.layers.Dense(num_classes, activation='softmax',
                                name='predictions')(merged)

    return tf.keras.Model(inputs=[ecg_in, rr_in], outputs=out, name='Tarang_ECG_v4')


model = build_tarang_cnn_v4(num_classes=len(le.classes_))
model.summary()

p = model.count_params()
print(f"\nParameters: {p:,}")
print(f"Est INT8  : ~{p/1024:.1f} KB  ({'PASS' if p < 51200 else 'OVER 50KB'})")

In [ ]:
class SparseFocalLoss(tf.keras.losses.Loss):
    """
    Focal loss for class-imbalanced ECG.
    get_config() included — load_model works without custom_objects.
    Use load_weights instead (faster, no serialisation issues).
    """
    def __init__(self, alpha, gamma=2.0, **kwargs):
        super().__init__(**kwargs)
        self._alpha_list = list(alpha)
        self.alpha = tf.constant(alpha, dtype=tf.float32)
        self.gamma = gamma

    def call(self, y_true, y_pred):
        y_true   = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_pred   = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        y_onehot = tf.one_hot(y_true, depth=tf.shape(y_pred)[-1])
        ce       = -y_onehot * tf.math.log(y_pred)
        pt       = tf.reduce_sum(y_onehot * y_pred, axis=-1)
        fw       = tf.pow(1.0 - pt, self.gamma)
        aw       = tf.reduce_sum(self.alpha * y_onehot, axis=-1)
        return tf.reduce_mean(aw * fw * tf.reduce_sum(ce, axis=-1))

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'alpha': self._alpha_list, 'gamma': self.gamma})
        return cfg

# label order: le.classes_ is alphabetical → N=0, S=1, V=2
# v4: Dialed back from v3's [0.10, 0.60, 0.30] — v3 overshot and S F1 *dropped* to 0.10
# v2 had alpha=0.50 for S with S F1=0.16; we go slightly below + pair with normalization fix
focal_alpha = [0.15, 0.45, 0.40]
focal_gamma = 2.0
print(f"Focal Loss: alpha={focal_alpha} (N, S, V), gamma={focal_gamma}")
print(f"Classes   : {list(le.classes_)}")
print(f"Note: v3 used [0.10, 0.60, 0.30] — caused training collapse with 15x S aug")

In [ ]:
# ── MacroF1Callback — monitors what actually matters for S/V quality ──────────
class MacroF1Callback(tf.keras.callbacks.Callback):
    """
    Computes macro F1 on the validation set at each epoch end.
    Logs as 'val_macro_f1' so EarlyStopping/ModelCheckpoint can monitor it.
    """
    def __init__(self, val_data, **kwargs):
        super().__init__(**kwargs)
        self.val_x, self.val_y = val_data

    def on_epoch_end(self, epoch, logs=None):
        y_pred_probs = self.model.predict(self.val_x, batch_size=128, verbose=0)
        y_pred = np.argmax(y_pred_probs, axis=1)
        mf1 = f1_score(self.val_y, y_pred, average='macro', zero_division=0)
        logs['val_macro_f1'] = mf1
        print(f"  — val_macro_f1: {mf1:.4f}")


# ── LR warmup + cosine decay schedule ────────────────────────────────────────
WARMUP_EPOCHS = 2
MAX_LR        = 5e-4
TOTAL_EPOCHS  = 150

def lr_schedule(epoch):
    """2-epoch linear warmup then cosine decay to 1e-6."""
    if epoch < WARMUP_EPOCHS:
        return MAX_LR * (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(TOTAL_EPOCHS - WARMUP_EPOCHS, 1)
    return 1e-6 + 0.5 * (MAX_LR - 1e-6) * (1 + np.cos(np.pi * progress))

print("LR schedule: 2-epoch warmup → cosine decay")
for e in [0, 1, 2, 10, 50, 100, 149]:
    print(f"  epoch {e:>3d}: lr={lr_schedule(e):.6f}")


# ── Compile with gradient clipping ───────────────────────────────────────────
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=MAX_LR, clipnorm=1.0),
    loss=SparseFocalLoss(alpha=focal_alpha, gamma=focal_gamma),
    metrics=['accuracy']
)

macro_f1_cb = MacroF1Callback(val_data=([X_val, X_rr_val], y_val))

callbacks = [
    macro_f1_cb,
    tf.keras.callbacks.EarlyStopping(
        monitor='val_macro_f1', patience=20, mode='max',
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.LearningRateScheduler(lr_schedule, verbose=0),
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_CHECKPOINT, monitor='val_macro_f1', mode='max',
        save_best_only=True, verbose=1
    ),
]

history = model.fit(
    [X_train_aug, X_rr_train_aug], y_train_aug,
    validation_data=([X_val, X_rr_val], y_val),
    epochs=TOTAL_EPOCHS,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

# float32 fix — numpy values aren't JSON serializable directly
with open(f'{OUTPUTS_DIR}/training_history.json', 'w') as f:
    json.dump({k: [float(x) for x in v] for k, v in history.history.items()}, f)
print(f"History saved.")

In [ ]:
print('='*60)
print('THRESHOLD TUNING — validation set (V-safety-first)')
print('='*60)

# load_weights: rebuilds architecture, loads weights only
# Avoids SparseFocalLoss deserialization entirely
model = build_tarang_cnn_v4(num_classes=len(le.classes_))
model.load_weights(MODEL_CHECKPOINT)

y_val_probs = model.predict([X_val, X_rr_val], batch_size=128, verbose=0)

n_idx = int(np.where(le.classes_ == 'N')[0])
s_idx = int(np.where(le.classes_ == 'S')[0])
v_idx = int(np.where(le.classes_ == 'V')[0])

V_RECALL_FLOOR  = 0.92   # back to 0.92 — 0.85 caused N:49% recall
best_macro_f1   = 0.0
best_thresholds = {'S': 0.5, 'V': 0.5}
best_v_recall   = 0.0

for t_v in np.arange(0.10, 0.55, 0.02):
    for t_s in np.arange(0.10, 0.60, 0.02):
        y_pred = np.full(len(y_val), n_idx, dtype=int)
        y_pred[y_val_probs[:, s_idx] > t_s] = s_idx
        y_pred[y_val_probs[:, v_idx] > t_v] = v_idx   # V overrides S — safety

        v_recall = (np.sum((y_val==v_idx) & (y_pred==v_idx))
                    / max(np.sum(y_val==v_idx), 1))
        if v_recall < V_RECALL_FLOOR:
            continue

        macro = f1_score(y_val, y_pred, average='macro', zero_division=0)
        if macro > best_macro_f1:
            best_macro_f1   = macro
            best_v_recall   = v_recall
            best_thresholds = {'S': round(t_s, 2), 'V': round(t_v, 2)}

print(f"Best thresholds: S>{best_thresholds['S']}  V>{best_thresholds['V']}")
print(f"Val Macro F1   : {best_macro_f1:.3f}")
print(f"V recall       : {best_v_recall:.3f}  (floor={V_RECALL_FLOOR})")

print('\n' + '='*60)
print('EVALUATION — TEST SET (MIT-BIH patient-wise held out)')
print('='*60)

y_pred_probs = model.predict([X_test, X_rr_test], batch_size=128, verbose=0)
y_pred_test  = np.full(len(y_test), n_idx, dtype=int)
y_pred_test[y_pred_probs[:, s_idx] > best_thresholds['S']] = s_idx
y_pred_test[y_pred_probs[:, v_idx] > best_thresholds['V']] = v_idx

print(classification_report(y_test, y_pred_test, target_names=le.classes_))

macro_f1  = f1_score(y_test, y_pred_test, average='macro')
per_class = f1_score(y_test, y_pred_test, average=None)
print(f"Macro F1: {macro_f1:.3f}")
for cls, f1 in zip(le.classes_, per_class):
    print(f"  F1 {cls}: {f1:.3f}  [{'PASS ✓' if f1>=0.85 else 'NEEDS WORK'}]")

print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_test))

In [ ]:
try:
    hist_data = history.history
except NameError:
    with open(f'{OUTPUTS_DIR}/training_history.json') as f:
        hist_data = json.load(f)

cm     = confusion_matrix(y_test, y_pred_test)
cm_pct = cm.astype('float') / cm.sum(axis=1)[:, None] * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[0])
axes[0].set_title('Confusion Matrix (counts)')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[1])
axes[1].set_title('Confusion Matrix (% of true class)')
axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')
plt.suptitle('Tarang ECG v4 — Test Set Performance', fontsize=13)
plt.tight_layout()
plt.savefig(f'{OUTPUTS_DIR}/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(hist_data['loss'],     label='Train')
axes[0].plot(hist_data['val_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(hist_data['accuracy'],     label='Train')
axes[1].plot(hist_data['val_accuracy'], label='Val')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
if 'val_macro_f1' in hist_data:
    axes[2].plot(hist_data['val_macro_f1'], label='Val Macro F1', color='#e74c3c')
    axes[2].set_title('Val Macro F1 (checkpoint metric)'); axes[2].legend(); axes[2].grid(True, alpha=0.3)
    axes[2].set_ylim([0, 1])
else:
    axes[2].text(0.5, 0.5, 'val_macro_f1 not in history', ha='center', va='center')
plt.tight_layout()
plt.savefig(f'{OUTPUTS_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def apply_run_filter(predictions, min_run=3, window=10):
    """Suppress isolated anomaly predictions — reduces false BLE wake events."""
    filtered = predictions.copy()
    for i in range(window, len(predictions)):
        if predictions[i] != n_idx and np.sum(predictions[i-window:i] != n_idx) < min_run:
            filtered[i] = n_idx
    return filtered

y_pred_filtered = apply_run_filter(y_pred_test)

print("=== BLE Run Filter ===")
print(f"Before: Macro F1 = {f1_score(y_test, y_pred_test,     average='macro'):.3f}")
print(f"After : Macro F1 = {f1_score(y_test, y_pred_filtered,  average='macro'):.3f}")
for cls, idx_ in [('N',n_idx),('V',v_idx),('S',s_idx)]:
    b = np.sum((y_test==idx_)&(y_pred_test==idx_))    / max(np.sum(y_test==idx_),1)
    a = np.sum((y_test==idx_)&(y_pred_filtered==idx_)) / max(np.sum(y_test==idx_),1)
    print(f"  {cls} recall: {b:.3f} → {a:.3f}")
fa_b = np.sum((y_test==n_idx)&(y_pred_test!=n_idx))
fa_a = np.sum((y_test==n_idx)&(y_pred_filtered!=n_idx))
print(f"False BLE alerts: {fa_b} → {fa_a}  ({(fa_b-fa_a)/max(fa_b,1)*100:.1f}% reduction)")

In [ ]:
print("=== Peak Jitter Degradation Test ===")
print("Simulates Pan-Tompkins misalignment on dry-pad ECG\n")

def jitter_macro_f1(mdl, X_ecg, X_rr, y, jitter_std, thresholds):
    if jitter_std == 0:
        Xj = X_ecg
    else:
        j  = np.random.normal(0, jitter_std, len(X_ecg)).astype(int)
        Xj = np.array([np.roll(X_ecg[i], j[i], axis=0) for i in range(len(X_ecg))])
    probs = mdl.predict([Xj, X_rr], batch_size=128, verbose=0)
    preds = np.full(len(y), n_idx, dtype=int)
    preds[probs[:, s_idx] > thresholds['S']] = s_idx
    preds[probs[:, v_idx] > thresholds['V']] = v_idx
    return f1_score(y, preds, average='macro', zero_division=0)

base = jitter_macro_f1(model, X_test, X_rr_test, y_test, 0, best_thresholds)
print(f"{'Jitter':>10}  {'F1':>8}  {'Drop':>8}  Status")
print("-"*48)
for j in [0, 5, 10, 15, 20]:
    f1   = jitter_macro_f1(model, X_test, X_rr_test, y_test, j, best_thresholds)
    drop = base - f1
    print(f"  ±{j:>2} smp   {f1:.3f}   {drop:.3f}   {'OK' if drop<0.05 else 'WIDEN SHIFT'}")

In [ ]:
# RR_MEAN/RR_STD already computed in Cell 8 — print for firmware reference
feature_names = ['rr_prev','rr_next','rr_ratio','rr_mean_5','rr_std_5']
print("RR normalization stats (from Cell 8):")
for i, name in enumerate(feature_names):
    print(f"  {name}: mean={RR_MEAN[i]:.5f}  std={RR_STD[i]:.5f}")

model = build_tarang_cnn_v4(num_classes=len(le.classes_))
model.load_weights(MODEL_CHECKPOINT)

# v4: X_rr_train is ALREADY normalized (done in Cell 8)
# No double-normalization — representative dataset uses data as-is
def representative_dataset():
    idx = np.random.choice(len(X_train), min(1000, len(X_train)), replace=False)
    for i in idx:
        ecg_s = X_train[i:i+1].astype(np.float32)
        rr_s  = X_rr_train[i:i+1].astype(np.float32)  # already normalized in Cell 8
        yield [ecg_s, rr_s]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations              = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset     = representative_dataset
converter.target_spec.supported_ops  = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type       = tf.int8
converter.inference_output_type      = tf.int8

print("\nQuantizing to INT8...")
tflite_model = converter.convert()
print("Done.")

with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

size_kb = len(tflite_model) / 1024
print(f"\nINT8 size: {size_kb:.1f} KB  ({'PASS ✓' if size_kb<50 else 'FAIL — trim model'})")

interp = tf.lite.Interpreter(model_content=tflite_model)
interp.allocate_tensors()
in_details  = interp.get_input_details()
out_details = interp.get_output_details()

# Identify ECG vs RR tensor by shape
ecg_det = next(d for d in in_details if d['shape'][1] == WINDOW_LEN)
rr_det  = next(d for d in in_details if d['shape'][1] == 5)
out_det = out_details[0]

ecg_scale, ecg_zp = ecg_det['quantization']
rr_scale,  rr_zp  = rr_det['quantization']
out_scale, out_zp  = out_det['quantization']

print(f"ECG input  : scale={ecg_scale:.6f}  zp={ecg_zp}  (expect ~0.031)")
print(f"RR input   : scale={rr_scale:.6f}  zp={rr_zp}")
print(f"Output     : scale={out_scale:.6f}  zp={out_zp}")

if ecg_scale > 0.1:
    print("WARNING: ECG scale too high — check representative_dataset normalization")

# INT8 accuracy verification
# v4: X_rr_test is already normalized in Cell 8 — use directly
n        = min(2000, len(X_test))
X_e_samp = X_test[:n]
X_r_samp = X_rr_test[:n].astype(np.float32)  # already normalized
y_samp   = y_test[:n]

float_preds = np.argmax(model.predict([X_e_samp, X_r_samp], batch_size=128, verbose=0), axis=1)
float_acc   = np.mean(float_preds == y_samp) * 100

correct = 0
for i in range(n):
    xeq = np.clip(np.round(X_e_samp[i:i+1]/ecg_scale+ecg_zp),-128,127).astype(np.int8)
    xrq = np.clip(np.round(X_r_samp[i:i+1]/rr_scale +rr_zp), -128,127).astype(np.int8)
    interp.set_tensor(ecg_det['index'], xeq)
    interp.set_tensor(rr_det['index'],  xrq)
    interp.invoke()
    out_f = (interp.get_tensor(out_det['index']).astype(np.float32) - out_zp) * out_scale
    if np.argmax(out_f) == y_samp[i]:
        correct += 1

int8_acc = correct / n * 100
drop     = float_acc - int8_acc
print(f"\nFloat32 acc: {float_acc:.2f}%")
print(f"INT8    acc: {int8_acc:.2f}%")
print(f"Drop       : {drop:.2f}%  ({'PASS ✓' if drop<3 else 'FAIL — check normalization'})")

print(f"\n{'='*55}")
print("COPY THESE DEFINES INTO FIRMWARE")
print('='*55)
print(f"#define ECG_INPUT_SCALE        {ecg_scale:.8f}f")
print(f"#define ECG_INPUT_ZERO_POINT   {ecg_zp}")
print(f"#define RR_INPUT_SCALE         {rr_scale:.8f}f")
print(f"#define RR_INPUT_ZERO_POINT    {rr_zp}")
print(f"#define OUTPUT_SCALE           {out_scale:.8f}f")
print(f"#define OUTPUT_ZERO_POINT      {out_zp}")
print(f"#define CONFIDENCE_THRESH_INT8 {int(round(CONFIDENCE_THRESHOLD/out_scale+out_zp))}")
for i, name in enumerate(feature_names):
    print(f"#define RR_MEAN_{name.upper():<14} {RR_MEAN[i]:.6f}f")
for i, name in enumerate(feature_names):
    print(f"#define RR_STD_{name.upper():<15} {RR_STD[i]:.6f}f")

In [ ]:
try:
    result = subprocess.run(['xxd', '-i', TFLITE_PATH], capture_output=True, text=True)
    xxd_ok = result.returncode == 0 and bool(result.stdout)
except FileNotFoundError:
    xxd_ok = False

if xxd_ok:
    with open(HEADER_PATH, 'w') as f:
        f.write('// Tarang ECG CNN v3 — INT8 dual-input (ECG + RR)\n// Auto-generated\n\n')
        f.write('#pragma once\n\n')
        f.write(result.stdout)
    print(f"Generated {HEADER_PATH} (xxd)")
else:
    with open(TFLITE_PATH, 'rb') as f:
        data = f.read()
    hex_lines = ['  ' + ', '.join(f'0x{b:02x}' for b in data[i:i+12]) + ','
                 for i in range(0, len(data), 12)]
    with open(HEADER_PATH, 'w') as f:
        f.write('// Tarang ECG CNN v3 — INT8 dual-input (ECG + RR)\n// Auto-generated\n\n')
        f.write('#pragma once\n\n')
        f.write(f'#define TARANG_MODEL_LEN {len(data)}\n\n')
        f.write('alignas(8) const unsigned char tarang_int8_tflite[] = {\n')
        f.write('\n'.join(hex_lines))
        f.write('\n};\n')
    print(f"Generated {HEADER_PATH} ({len(data):,} bytes, Python fallback — xxd not found)")

## Firmware Integration Notes

### `tarang_ai_process()` — call signature

```c
// tarang_ai.h
typedef struct {
    float rr_prev;    // seconds, from Pan-Tompkins RR buffer
    float rr_next;    // seconds (use 0.0f if next peak not yet arrived — model handles gracefully)
    float rr_ratio;   // rr_prev / rr_next
    float rr_mean_5;  // mean of ±2 local RR intervals (seconds)
    float rr_std_5;   // std  of ±2 local RR intervals (seconds)
} tarang_rr_features_t;

tarang_label_t tarang_ai_process(
    const int8_t  *ecg_window_q,   // 187 samples × 2 channels, INT8-quantized, z-score normalised
    const float   *rr_raw,         // rr_features[5], RAW (not yet normalised)
    tarang_conf_t *conf_out
);
```

### RR normalization before quantization

```c
// Step 1: normalize  →  x_norm = (x_raw - RR_MEAN_xxx) / RR_STD_xxx
// Step 2: quantize   →  x_q = clamp(round(x_norm / RR_INPUT_SCALE + RR_INPUT_ZERO_POINT), -128, 127)
// Constants are printed by the quantization cell above.
```

### Two-input TFLite Micro invocation

```c
// Tensor indices — check tarang_model.h or print interp.get_input_details()
TfLiteTensor *ecg_tensor = interpreter->input(ECG_TENSOR_IDX);
TfLiteTensor *rr_tensor  = interpreter->input(RR_TENSOR_IDX);

// ECG: 187 samples × 2 channels
memcpy(ecg_tensor->data.int8, ecg_window_q, WINDOW_LEN * 2 * sizeof(int8_t));
// RR: 5 features, normalised then quantised
memcpy(rr_tensor->data.int8,  rr_q,         5 * sizeof(int8_t));

TfLiteStatus status = interpreter->Invoke();
```

### Pan-Tompkins RR buffer
Your existing Pan-Tompkins already maintains an RR ring buffer.
You need a 5-element window around the current beat.
`rr_next = 0.0f` is safe when the next peak hasn't arrived — the model degrades gracefully
since `rr_prev`, `rr_ratio`, and `rr_mean_5` carry most S-class discriminative power.